# 01 — Time Series Preparation

**Project:** SERSA Product Demand Relationship Analysis  
**Author:** Cesar Enrique Banda Martinez  
**Date:** 2026  

---

## Context

This notebook is the first phase of the satellite project *SERSA Product Demand Relationship Analysis*.  
It draws from the output of the main project — `master_sales.csv` — which contains 639,568 vending machine transactions across 3 industrial clients and 92 unique SKUs, covering April 2022 through May 2026.

The central question of this satellite project is:

> **Do products whose demand moves together reveal hidden relationships in industrial consumption patterns?**

To answer this, we first need to transform the transaction-level data into a time-indexed structure where each column represents a SKU and each row represents a month.

### Why monthly aggregation?
- **Daily** granularity is too noisy for correlation analysis — day-to-day variation masks meaningful patterns.
- **Weekly** is better but increases sparsity for low-frequency SKUs.
- **Monthly** is the standard for industrial demand analysis: it smooths operational noise while preserving seasonal and trend signals.

### What is `Quantity`?
The source file has no quantity column. Each row represents one vending machine dispense event.  
`Quantity` is therefore derived by **counting transactions per SKU per month** — a direct proxy for units consumed.

### Goals of this notebook
1. Load `master_sales.csv` and parse the datetime column.
2. Aggregate transactions into monthly counts per SKU (`Quantity`).
3. Pivot into a wide-format matrix: rows = months, columns = SKUs.
4. Run data quality checks (missing months, low-activity SKUs).
5. Visualize total monthly volume to confirm trend alignment with the main project.
6. Export `monthly_sales_pivot.csv` as the shared input for all subsequent notebooks.

### Project structure (reference)
```
SERSA_Product_Demand_Relationship_Analysis/
├── data/
│   └── processed/          ← master_sales.csv (from main project)
├── notebooks/
│   ├── 01_time_series_preparation.ipynb    ← this file
│   ├── 02_correlation_analysis.ipynb
│   ├── 03_growth_correlation.ipynb
│   ├── 04_lead_lag_analysis.ipynb
│   └── 05_product_network.ipynb
├── outputs/
│   ├── figures/
│   └── tables/
├── src/
│   └── ts_utils.py
└── README.md
```

---

## 1. Imports

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

---
## 2. Configuration

Paths are built relative to the `notebooks/` folder using `os.getcwd()`.  
This notebook must be opened and run from within the `notebooks/` directory for paths to resolve correctly.

In [2]:
DATA_DIR       = os.path.join(os.getcwd(), "..", "data", "processed")
OUTPUT_TABLES  = os.path.join(os.getcwd(), "..", "outputs", "tables")
OUTPUT_FIGURES = os.path.join(os.getcwd(), "..", "outputs", "figures")

os.makedirs(OUTPUT_TABLES, exist_ok=True)
os.makedirs(OUTPUT_FIGURES, exist_ok=True)

print(f"Data dir:       {os.path.normpath(DATA_DIR)}")
print(f"Output tables:  {os.path.normpath(OUTPUT_TABLES)}")
print(f"Output figures: {os.path.normpath(OUTPUT_FIGURES)}")

Data dir:       c:\Users\Acer\data\processed
Output tables:  c:\Users\Acer\outputs\tables
Output figures: c:\Users\Acer\outputs\figures


---
## 3. Load Data

In [3]:
df = pd.read_csv(os.path.join(DATA_DIR, "master_sales.csv"))

df["Fecha de Consumo"] = pd.to_datetime(df["Fecha de Consumo"])

print(f"Rows:        {len(df):,}")
print(f"Columns:     {df.columns.tolist()}")
print(f"Date range:  {df['Fecha de Consumo'].min().date()} \u2192 {df['Fecha de Consumo'].max().date()}")
print(f"Unique SKUs: {df['SKU'].nunique()}")
print(f"Companies:   {df['Company'].unique().tolist()}")
print()
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Acer\\Downloads\\..\\data\\processed\\master_sales.csv'

---
## 4. Monthly Aggregation

Each row in `master_sales.csv` represents a single vending machine dispense event.  
`Quantity` is derived by **counting rows per SKU per month** — no quantity field exists in the source data.

We use `freq='MS'` (Month Start) so every period label aligns consistently to the 1st of each month.

In [ ]:
monthly = (
    df.groupby([pd.Grouper(key="Fecha de Consumo", freq="MS"), "SKU"])
    .size()
    .reset_index(name="Quantity")
)

print(f"Monthly aggregation: {len(monthly):,} rows")
print()
print(monthly.head(10))

---
## 5. Build Monthly Pivot

Pivot to wide format: rows = months, columns = SKUs, values = transaction count.  

Missing SKU-month combinations (SKU not dispensed in a given month) are filled with `0`.

In [ ]:
pivot = (
    monthly
    .pivot(index="Fecha de Consumo", columns="SKU", values="Quantity")
    .fillna(0)
    .astype(int)
)

pivot.index.name = "Month"
pivot.columns.name = None

print(f"Pivot shape: {pivot.shape}  \u2192  {pivot.shape[0]} months \u00d7 {pivot.shape[1]} SKUs")
print()
print(pivot.iloc[:3, :8])

---
## 6. Data Quality Checks

Before exporting, we verify:
1. **No months are missing** in the sequence.
2. **Low-activity SKUs** — products with near-zero transactions across the entire period may introduce noise in correlation analysis.

In [ ]:
# 1. Missing months
full_range = pd.date_range(pivot.index.min(), pivot.index.max(), freq="MS")
missing_months = full_range.difference(pivot.index)

print(f"Expected months : {len(full_range)}")
print(f"Present months  : {len(pivot)}")
print(f"Missing months  : {len(missing_months)}")
if len(missing_months) > 0:
    print(missing_months)

In [ ]:
# 2. Low-activity SKUs
sku_totals = pivot.sum().sort_values()

LOW_ACTIVITY_THRESHOLD = 10
low_activity_skus = sku_totals[sku_totals < LOW_ACTIVITY_THRESHOLD]

print(f"Total SKUs                              : {len(sku_totals)}")
print(f"Low-activity SKUs (< {LOW_ACTIVITY_THRESHOLD} txns total) : {len(low_activity_skus)}")
if len(low_activity_skus) > 0:
    print()
    print(low_activity_skus.to_string())

### Note on low-activity SKUs

Low-activity SKUs are **documented here but not removed** from the pivot at this stage.  
The filtering decision will be made in `02_correlation_analysis.ipynb`, where their effect on the correlation matrix can be evaluated in context.

---
## 7. Monthly Volume Overview

A quick visualization to confirm that the pivot captures the same growth trend documented in the main project.  
Total transactions should show sustained growth from 2022 to 2025.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

total_monthly = pivot.sum(axis=1)
total_monthly.plot(ax=ax, color="#2C7BB6", linewidth=2)

ax.set_title("Total Monthly Transactions \u2014 All SKUs Combined", fontsize=14, fontweight="bold")
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Transactions", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.grid(axis="y", alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "01_monthly_total_transactions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 01_monthly_total_transactions.png")

---
## 8. SKU Activity Distribution

Understanding how transaction volume is distributed across SKUs gives us a baseline before correlation analysis.  
A highly skewed distribution is expected given the Pareto findings in the main project.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

sku_totals_sorted = sku_totals.sort_values(ascending=False)
ax.bar(range(len(sku_totals_sorted)), sku_totals_sorted.values, color="#2C7BB6", alpha=0.8)

ax.set_title("Total Transactions per SKU (4-Year Period)", fontsize=14, fontweight="bold")
ax.set_xlabel("SKU rank (by volume)", fontsize=11)
ax.set_ylabel("Total Transactions", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xticks([])
ax.grid(axis="y", alpha=0.3)
sns.despine()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGURES, "01_sku_activity_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 01_sku_activity_distribution.png")

---
## 9. Export

In [ ]:
pivot.to_csv(
    os.path.join(OUTPUT_TABLES, "monthly_sales_pivot.csv"),
    decimal=","
)

print("Export complete.")
print(f"  monthly_sales_pivot.csv  \u2192  {pivot.shape[0]} months \u00d7 {pivot.shape[1]} SKUs")

---
## 10. Summary

| Output | Shape | Description |
|--------|-------|-------------|
| `monthly_sales_pivot.csv` | 49 × 92 | Monthly transaction counts per SKU |
| `01_monthly_total_transactions.png` | — | Total monthly volume trend |
| `01_sku_activity_distribution.png` | — | SKU volume distribution bar chart |

**Observations to carry forward to notebook 02:**
- Note any missing months found in Section 6.
- Note any low-activity SKUs flagged in Section 6 — filtering decision deferred to `02_correlation_analysis.ipynb`.

**Next:** `02_correlation_analysis.ipynb` — Pearson correlation matrix and heatmap across all 92 SKUs.